# 07 — Stage 2: Fusion + Multi-label Genre Classifier

Needs: Stage 1 embeddings + rhythm/timbre/harmony CSVs.

- Fusion A: concat → linear
- Fusion B: single-head attention over the 4 concept tokens
- Head: 87-ish genre tags, BCE with logits
- Report **split-0 test** only


## Kaggle setup (every notebook)

### A. Settings
1. Right sidebar → **Internet → On** (required for downloads).
2. **GPU**: Off for `00`/`01`/`04`–`06`. **GPU (T4)** on for `02`/`03`/`07`.

### B. How data moves (do not skip)
Kaggle **does not** keep `/kaggle/working` when you open a *new* notebook.

**After notebook 00 finishes:**
1. **Save Version** (top-right) → **Save & Run All** (or Quick Save if already finished).
2. Open **Advanced** → tick **Always save output**.
3. Wait until the version is **Success**.
4. Note the kernel slug (yours is **`thevifernando/dnn-download-data-1`**).

**In the next notebook (01, then 02, …):**
1. **Add Input** (right sidebar) → **Your notebooks** / **Notebook Output**.
2. Select **`dnn-download-data-1`** (latest successful version).
3. Files appear at `/kaggle/input/dnn-download-data-1/` (**read-only**).
4. This bootstrap **reads mels from that input** (does **not** copy 10 shards — they would overflow disk).
5. It **writes** new files (manifest, features, checkpoints) to `/kaggle/working/MTG_Instrument`.
6. **Save Version + save output** again so the *next* notebook can **Add Input** *this* notebook too (chain: 00 → 01 → 02 …).

### C. CLI (laptop only — not needed on Kaggle)
```bash
kaggle kernels output thevifernando/dnn-download-data-1 -p ./from_00
```
On Kaggle you **Add Input** instead of this command.

### D. GitHub
Commit **notebooks only** to `thevindu-branch`. Do **not** git-push the `.npy` shards (too large). Data stays on Kaggle output.


## Step 0 — Packages (enable GPU)


In [ ]:
!pip install -q scikit-learn tqdm


## Step 1 — Bootstrap paths


In [ ]:
from pathlib import Path
import os, json, random, re, shutil, socket, time, urllib.request
import numpy as np
import pandas as pd

KERNEL_SLUG = "dnn-download-data-1"  # notebook 00 Kaggle slug — change if yours differs
WORKING_ROOT = Path("/kaggle/working/MTG_Instrument")
INPUT_BASE = Path("/kaggle/input")
RAW_ANN = "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data"
NEEDED_ANN = [
    "splits/split-0/autotagging_genre-train.tsv",
    "splits/split-0/autotagging_genre-validation.tsv",
    "splits/split-0/autotagging_genre-test.tsv",
    "splits/split-0/autotagging_instrument-train.tsv",
    "splits/split-0/autotagging_instrument-validation.tsv",
    "splits/split-0/autotagging_instrument-test.tsv",
    "autotagging_genre.tsv",
    "autotagging_instrument.tsv",
]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def check_internet(host: str = "github.com", port: int = 443, timeout: float = 5) -> bool:
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


def normalize_track_id(raw) -> str | None:
    """MTG ids are 7-digit zero-padded (track_0000948 → 0000948)."""
    m = re.search(r"(\d+)", str(raw))
    if not m:
        return None
    return f"{int(m.group(1)):07d}"


def _find_file(name: str, bases: list[Path]) -> Path | None:
    for base in bases:
        if not base.exists():
            continue
        hits = list(base.rglob(name))
        if hits:
            return hits[0]
    return None


def discover_input_root() -> Path | None:
    """Find a previous notebook-00 output or MTG data folder under /kaggle/input."""
    if not INPUT_BASE.exists():
        return None
    for marker in [
        "song_manifest.csv",
        "autotagging_genre-train.tsv",
        "autotagging_genre.tsv",
        ".shard_00_done",
    ]:
        hit = _find_file(marker, [INPUT_BASE])
        if hit is None:
            continue
        if marker == "song_manifest.csv":
            return hit.parents[1]  # .../MTG_Instrument/dataset/song_manifest.csv
        if marker == "autotagging_genre-train.tsv":
            # .../annotations/splits/split-0/file  OR  .../data/splits/split-0/file
            p = hit
            for _ in range(6):
                if (p / "dataset").exists() or p.name in {"MTG_Instrument", "data"}:
                    return p if p.name != "data" else p
                p = p.parent
            return hit.parents[2]
        if marker == "autotagging_genre.tsv":
            parent = hit.parent
            if parent.name == "annotations":
                return parent.parent
            return parent  # MTG data/
        if marker == ".shard_00_done":
            return hit.parents[2]  # .../MTG_Instrument/dataset/logmel_songs/.shard
    for p in INPUT_BASE.rglob("MTG_Instrument"):
        if p.is_dir():
            return p
    return None


def find_mel_dir() -> Path:
    """Prefer attached kernel output (read-only). Never copy 10 shards into working."""
    bases = [
        Path(f"/kaggle/input/{KERNEL_SLUG}") / "MTG_Instrument" / "dataset" / "logmel_songs",
        Path(f"/kaggle/input/{KERNEL_SLUG}") / "dataset" / "logmel_songs",
        WORKING_ROOT / "dataset" / "logmel_songs",
    ]
    kernel = Path(f"/kaggle/input/{KERNEL_SLUG}")
    extra = []
    if INPUT_BASE.exists():
        extra.append(INPUT_BASE)
    if kernel.exists():
        extra.append(kernel)
    for b in bases:
        if b.exists() and next(b.rglob("*.npy"), None) is not None:
            return b
    for b in extra:
        hit = next(b.rglob("*.npy"), None) if b.exists() else None
        if hit is None:
            continue
        p = hit.parent
        for _ in range(6):
            if p.name == "logmel_songs":
                return p
            p = p.parent
        return hit.parent
    return WORKING_ROOT / "dataset" / "logmel_songs"


def ensure_annotations(ann_dir: Path) -> Path:
    """Make sure split-0 TSVs exist; wget them if this is a fresh Kaggle session."""
    train = ann_dir / "splits" / "split-0" / "autotagging_genre-train.tsv"
    if train.exists():
        return ann_dir

    # maybe files are flat, or under /kaggle/input with a different layout
    hit = _find_file("autotagging_genre-train.tsv", [ann_dir, INPUT_BASE, Path("/kaggle/working")])
    if hit is not None:
        dest = ann_dir / "splits" / "split-0" / hit.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if hit.resolve() != dest.resolve():
            shutil.copy2(hit, dest)
        # copy sibling split files from the same folder
        for name in [
            "autotagging_genre-validation.tsv",
            "autotagging_genre-test.tsv",
            "autotagging_instrument-train.tsv",
            "autotagging_instrument-validation.tsv",
            "autotagging_instrument-test.tsv",
        ]:
            sib = hit.parent / name
            if sib.exists():
                shutil.copy2(sib, dest.parent / name)
        genre_full = _find_file("autotagging_genre.tsv", [hit.parents[2] if len(hit.parents) > 2 else hit.parent, INPUT_BASE])
        if genre_full:
            shutil.copy2(genre_full, ann_dir / "autotagging_genre.tsv")
        inst_full = _find_file("autotagging_instrument.tsv", [hit.parents[2] if len(hit.parents) > 2 else hit.parent, INPUT_BASE])
        if inst_full:
            shutil.copy2(inst_full, ann_dir / "autotagging_instrument.tsv")
        print("Recovered split files from", hit.parent)
        return ann_dir

    if not check_internet():
        raise FileNotFoundError(
            "Split TSVs not found and Internet is OFF.\n"
            "Do ONE of:\n"
            "  A) Settings → Internet → On, re-run this cell (auto-download)\n"
            "  B) Add Data → attach notebook-00 output dataset (mtg-instrument-cache)\n"
            "  C) Stay in the SAME Kaggle session after running notebook 00"
        )

    print("Split TSVs missing — downloading official MTG annotations...")
    n = 0
    for rel in NEEDED_ANN:
        dest = ann_dir / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        url = f"{RAW_ANN}/{rel}"
        print("  wget", url)
        urllib.request.urlretrieve(url, dest)
        n += 1
    print(f"Downloaded {n} annotation files into {ann_dir}")
    return ann_dir


def load_split_ids(split: str, subset: str = "genre") -> set[str]:
    candidates = [
        ANN_DIR / "splits" / "split-0" / f"autotagging_{subset}-{split}.tsv",
        ANN_DIR / f"autotagging_{subset}-{split}.tsv",
        ANN_DIR / "splits" / "split-0" / f"{split}.tsv",
        ANN_DIR / f"{split}.tsv",
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        found = _find_file(f"autotagging_{subset}-{split}.tsv", [ANN_DIR, INPUT_BASE, Path("/kaggle/working")])
        path = found
    if path is None:
        raise FileNotFoundError(
            f"No split file for {subset}/{split}.\n"
            "Re-run the bootstrap cell after enabling Internet, or attach notebook-00 output."
        )
    df = pd.read_csv(path, sep="\t")
    col = "TRACK_ID" if "TRACK_ID" in df.columns else df.columns[0]
    ids = set()
    for v in df[col].astype(str):
        tid = normalize_track_id(v)
        if tid:
            ids.add(tid)
    print(f"{split:12s}  {len(ids):6d} ids   ← {path}")
    return ids


MEL_CACHE = Path("/kaggle/working/mel_cache")
MEL_CACHE.mkdir(parents=True, exist_ok=True)


def load_mel_npy(mel_abs, retries=5, pause=1.0):
    """Load mel with retries; cache under /kaggle/working for stable re-reads."""
    mel_abs = Path(mel_abs)
    sid = normalize_track_id(mel_abs.stem) or mel_abs.stem.replace("/", "_")
    cached = MEL_CACHE / f"{sid}.npy"
    if cached.exists():
        try:
            return np.load(cached)
        except (OSError, ValueError):
            cached.unlink(missing_ok=True)

    last_err = None
    for attempt in range(retries):
        try:
            arr = np.load(mel_abs, mmap_mode=None)
            arr = np.asarray(arr, dtype=np.float32)
            np.save(cached, arr)
            return arr
        except (OSError, ValueError) as e:
            last_err = e
            if attempt + 1 < retries:
                time.sleep(pause * (attempt + 1))
    nbytes = mel_abs.stat().st_size if mel_abs.exists() else "missing"
    raise RuntimeError(
        f"Bad/truncated mel — re-run notebook 00 for this shard: {mel_abs} "
        f"({nbytes} bytes). {last_err}"
    ) from last_err


def scan_bad_mels(df, label="manifest"):
    from tqdm.auto import tqdm

    bad = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"scan {label}"):
        try:
            load_mel_npy(row["mel_abs"])
        except Exception as e:
            bad.append({"song_id": str(row["song_id"]), "mel_abs": row["mel_abs"], "error": str(e)})
    if bad:
        out = RESULTS_DIR / f"bad_mels_{label}.json"
        out.write_text(json.dumps(bad, indent=2))
        print(f"WARNING: {len(bad)} bad mels → {out}")
    else:
        print(f"scan {label}: all {len(df)} mels OK (cache: {MEL_CACHE})")
    return bad


ONLINE = check_internet()
print("Internet reachable:", ONLINE)
print("KERNEL_SLUG =", KERNEL_SLUG)
print("/kaggle/input folders:", list(INPUT_BASE.iterdir()) if INPUT_BASE.exists() else "n/a")

ROOT = WORKING_ROOT
ROOT.mkdir(parents=True, exist_ok=True)
MEL_DIR = find_mel_dir()
ANN_DIR = ROOT / "annotations"
# if annotations only exist on the attached kernel, point there (read-only is OK)
for cand in [
    Path(f"/kaggle/input/{KERNEL_SLUG}") / "MTG_Instrument" / "annotations",
    Path(f"/kaggle/input/{KERNEL_SLUG}") / "annotations",
]:
    if (cand / "splits" / "split-0" / "autotagging_genre-train.tsv").exists():
        ANN_DIR = cand
        break
FEAT_DIR = ROOT / "features"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
MANIFEST = ROOT / "dataset" / "song_manifest.csv"
att_manifest = _find_file("song_manifest.csv", [INPUT_BASE, Path("/kaggle/working")])
if not MANIFEST.exists() and att_manifest is not None:
    MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    try:
        shutil.copy2(att_manifest, MANIFEST)
        print("Copied song_manifest.csv from", att_manifest)
    except OSError:
        MANIFEST = att_manifest

for p in [ROOT / "dataset", ROOT / "annotations", FEAT_DIR, CKPT_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Small TSVs: copy/wget into working. Large mels stay on /kaggle/input.
ANN_DIR = ensure_annotations(ROOT / "annotations")

print("ROOT     =", ROOT)
print("MEL_DIR  =", MEL_DIR, "npy=", len(list(MEL_DIR.rglob('*.npy'))))
print("ANN_DIR  =", ANN_DIR)
print("split-0 train exists:", (ANN_DIR / "splits/split-0/autotagging_genre-train.tsv").exists())
print("MANIFEST =", MANIFEST, "exists=", MANIFEST.exists())


## Step 2 — Load concept features + genre labels


In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not MANIFEST.exists():
    raise FileNotFoundError("Run notebooks 01 then 03–06 first.")
manifest = pd.read_csv(MANIFEST)
manifest["song_id"] = manifest["song_id"].astype(str).map(lambda s: normalize_track_id(s) or s)

inst_dir = FEAT_DIR / "instrument"
if not (inst_dir / "instrument_embeddings.npy").exists():
    raise FileNotFoundError("Missing Stage 1 embeddings — run notebook 03.")
E = np.load(inst_dir / "instrument_embeddings.npy")
inst_ids = json.loads((inst_dir / "song_ids.json").read_text())
inst_map = {normalize_track_id(s) or str(s): E[i] for i, s in enumerate(inst_ids)}

def load_feat(sub):
    p = FEAT_DIR / sub / f"{sub}_song.csv"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p} — run the matching 04/05/06 notebook.")
    df = pd.read_csv(p)
    df["song_id"] = df["song_id"].astype(str).map(lambda s: normalize_track_id(s) or s)
    return df.set_index("song_id")

rhythm, timbre, harmony = load_feat("rhythm"), load_feat("timbre"), load_feat("harmony")
if "source" in rhythm.columns and (rhythm["source"] == "mel_proxy").any():
    raise RuntimeError("rhythm_song.csv still has mel_proxy placeholders — re-run notebook 04 (AcousticBrainz)")

def num_cols(df):
    return [c for c in df.columns if c not in ("song_id", "source", "split") and pd.api.types.is_numeric_dtype(df[c])]

r_cols, t_cols, h_cols = num_cols(rhythm), num_cols(timbre), num_cols(harmony)
n_before = len(manifest)
have = set(inst_map) & set(rhythm.index) & set(timbre.index) & set(harmony.index)
manifest = manifest[manifest["song_id"].isin(have)].copy()
print(f"Stage 2 overlap: {len(manifest)} / {n_before} songs have instrument+rhythm+timbre+harmony")
if manifest.empty:
    raise RuntimeError("No overlapping songs — run 03–06 (04 must be AcousticBrainz)")
ids = manifest["song_id"].astype(str).tolist()
id_to_idx = {s: i for i, s in enumerate(ids)}

# genre Y
candidates = [ANN_DIR / "autotagging_genre.tsv", *ANN_DIR.rglob("*genre*.tsv")]
tag_to_idx, rows = {}, {s: set() for s in ids}
for path in candidates:
    if not Path(path).exists():
        continue
    df = pd.read_csv(path, sep="\t")
    id_col = "TRACK_ID" if "TRACK_ID" in df.columns else df.columns[0]
    tag_col = "TAGS" if "TAGS" in df.columns else df.columns[-1]
    for _, r in df.iterrows():
        sid = normalize_track_id(r[id_col])
        if sid not in rows:
            continue
        raw = r[tag_col]
        if pd.isna(raw):
            continue
        for tag in str(raw).replace("|", "\t").split("\t"):
            leaf = tag.strip().split("/")[-1].split("---")[-1]
            if not leaf or leaf.lower() in {"nan", "tags"}:
                continue
            tag_to_idx.setdefault(leaf, len(tag_to_idx))
            rows[sid].add(leaf)
    if tag_to_idx:
        print("genre tags from", path, len(tag_to_idx))
        break
TAG_NAMES = [None] * len(tag_to_idx)
for t, i in tag_to_idx.items():
    TAG_NAMES[i] = t
Y = np.zeros((len(ids), len(TAG_NAMES)), np.float32)
for i, sid in enumerate(ids):
    for t in rows[sid]:
        Y[i, tag_to_idx[t]] = 1.0
print("Y", Y.shape, "inst/rhythm/timbre/harmony dims", 64, len(r_cols), len(t_cols), len(h_cols))


## Step 3 — Fusion models (set `FUSION = "attention"` or `"linear"`)


In [ ]:
class ConceptDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        sid = str(self.df.iloc[i]["song_id"])
        inst = inst_map[sid].astype(np.float32)
        r = rhythm.loc[sid, r_cols].astype(np.float32).fillna(0).values if sid in rhythm.index else np.zeros(len(r_cols), np.float32)
        t = timbre.loc[sid, t_cols].astype(np.float32).fillna(0).values if sid in timbre.index else np.zeros(len(t_cols), np.float32)
        h = harmony.loc[sid, h_cols].astype(np.float32).fillna(0).values if sid in harmony.index else np.zeros(len(h_cols), np.float32)
        y = Y[id_to_idx[sid]]
        return torch.tensor(inst), torch.tensor(r), torch.tensor(t), torch.tensor(h), torch.tensor(y)

def loader(split, bs=32, shuffle=False):
    sub = manifest[manifest["split"] == split]
    assert set(sub["split"].unique()) == {split}
    return DataLoader(ConceptDataset(sub), batch_size=bs, shuffle=shuffle, num_workers=0)

class LinearFusion(nn.Module):
    def __init__(self, d_inst, d_r, d_t, d_h, fused=128, n_tags=87):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(d_inst + d_r + d_t + d_h, fused), nn.ReLU(), nn.Dropout(0.2))
        self.head = nn.Linear(fused, n_tags)

    def forward(self, inst, r, t, h):
        return self.head(self.proj(torch.cat([inst, r, t, h], -1))), None

class AttentionFusion(nn.Module):
    def __init__(self, d_inst, d_r, d_t, d_h, token=64, fused=128, n_tags=87):
        super().__init__()
        self.p_i, self.p_r, self.p_t, self.p_h = nn.Linear(d_inst, token), nn.Linear(d_r, token), nn.Linear(d_t, token), nn.Linear(d_h, token)
        self.attn = nn.MultiheadAttention(token, 1, batch_first=True)
        self.out = nn.Sequential(nn.Linear(token, fused), nn.ReLU(), nn.Dropout(0.2))
        self.head = nn.Linear(fused, n_tags)

    def forward(self, inst, r, t, h):
        tokens = torch.stack([self.p_i(inst), self.p_r(r), self.p_t(t), self.p_h(h)], 1)
        attn_out, w = self.attn(tokens, tokens, tokens, need_weights=True)
        return self.head(self.out(attn_out.mean(1))), w

FUSION = "attention"
dims = (64, len(r_cols), len(t_cols), len(h_cols))
Model = AttentionFusion if FUSION == "attention" else LinearFusion
model = Model(*dims, n_tags=Y.shape[1]).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.BCEWithLogitsLoss()
print("fusion", FUSION, "dims", dims)


## Step 4 — Train; keep best val PR-AUC; evaluate split-0 **test**


In [ ]:
def nan_safe(y_true, y_prob, kind="roc"):
    scores = []
    for k in range(y_true.shape[1]):
        if y_true[:, k].sum() in (0, len(y_true)):
            continue
        try:
            scores.append(roc_auc_score(y_true[:, k], y_prob[:, k]) if kind == "roc" else average_precision_score(y_true[:, k], y_prob[:, k]))
        except ValueError:
            continue
    return float(np.mean(scores)) if scores else float("nan")

@torch.no_grad()
def evaluate(dl):
    model.eval()
    ys, ps = [], []
    for inst, r, t, h, y in dl:
        inst, r, t, h = inst.to(DEVICE), r.to(DEVICE), t.to(DEVICE), h.to(DEVICE)
        logits, _ = model(inst, r, t, h)
        ps.append(torch.sigmoid(logits).cpu().numpy())
        ys.append(y.numpy())
    yt, yp = np.concatenate(ys), np.concatenate(ps)
    return {"macro_roc_auc": nan_safe(yt, yp, "roc"), "macro_pr_auc": nan_safe(yt, yp, "pr")}

train_dl, val_dl, test_dl = loader("train", shuffle=True), loader("validation"), loader("test")
best_macro_map = 0.0
ckpt = CKPT_DIR / "stage2"
ckpt.mkdir(parents=True, exist_ok=True)
hist = []
for epoch in range(1, 16):
    model.train()
    total = 0
    for inst, r, t, h, y in tqdm(train_dl, leave=False):
        inst, r, t, h, y = inst.to(DEVICE), r.to(DEVICE), t.to(DEVICE), h.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        logits, _ = model(inst, r, t, h)
        loss = crit(logits, y)
        loss.backward()
        opt.step()
        total += loss.item() * len(y)
    vm = evaluate(val_dl)
    hist.append({"epoch": epoch, "loss": total / len(train_dl.dataset), **vm})
    print(epoch, hist[-1])
    if vm["macro_pr_auc"] > best_macro_map:
        best_macro_map = vm["macro_pr_auc"]
        torch.save({"model": model.state_dict(), "fusion": FUSION, "best_macro_map": best_macro_map, "tags": TAG_NAMES}, ckpt / f"best_{FUSION}.pt")
        print("  ✓ saved", best_macro_map)

state = torch.load(ckpt / f"best_{FUSION}.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(state["model"])
test_m = evaluate(test_dl)
print("TEST split-0", test_m)
pd.DataFrame(hist).to_csv(RESULTS_DIR / f"07_stage2_{FUSION}_history.csv", index=False)
(RESULTS_DIR / f"07_stage2_{FUSION}_test.json").write_text(json.dumps(test_m, indent=2))
